# Point Sources, Airy Disks, and Resolution


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-18.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

`ch05-17` showed what one point source does: it images to an Airy disk of fixed
width and fixed shape, centered wherever that source lands. This notebook asks what
happens when there is more than one.

The answer is the whole reason convolution appears in imaging. Treat the object as a
row of point sources, each a delta function weighted by how bright it is. Every one of
them paints its own copy of the PSF, and the image is what you get when those copies
**add up**. That sum is exactly the convolution integral

$$I_\textrm{image}(Y) = \int I_\textrm{obj}(y)\, \left\lvert h(Y - M y) \right\rvert^2\, dy
= I_\textrm{obj} \otimes \left\lvert h \right\rvert^2$$

Once the copies start overlapping, two sources can stop looking like two. The second
figure pins down where that happens.


## Import libraries

The same `scipy.special.j1` that `ch05-17` used for the Airy pattern. `np.convolve`
appears only in the closing comparison; the figures build the image as an explicit sum
of shifted PSFs, because that sum is the thing being explained.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j1


In [ ]:
%matplotlib inline
# ==========================================
# 1. Configuration & Styling
# ==========================================
plot_params = {
    'font.size': 14,
    'font.family': 'sans-serif',
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.0,
    'lines.linewidth': 2.5,
}
plt.rcParams.update(plot_params)


def apply_custom_formatting(ax):
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.minorticks_on()
    ax.tick_params(which='both', direction="in", width=1)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=3.5)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')


## Define parameters

**The optics are the same as `ch05-17`**: a 25 mm aperture at 100 mm focal length in
green light, so $\lambda f / D = 2.2~\mu$m and the Airy radius is
$r_A = 1.22\,\lambda f / D = 2.68~\mu$m. Everything below is measured against that one
number, because it is the only length the instrument has.

**The object is a list of positions and weights.** `src_pos` says where each point
source images to and `src_amp` says how bright it is. This is the discrete version of
$I_\textrm{obj}$: a sum of weighted delta functions rather than a continuous
distribution.

**One pair is deliberately too close.** The sources at $4.0$ and $5.8~\mu$m are
$1.8~\mu$m apart, which is $0.67\,r_A$. That is below the separation at which two Airy
disks stop producing any dip between them, so they merge completely. Watch what they do
in the bottom panel, then see the second figure for why that threshold is where it is.

**The magnification is folded in already.** `src_pos` gives $M y$, not $y$, so the
object and image panels share one axis and the shift in $\left\lvert h(Y - My) \right\rvert^2$ is a
plain subtraction.


In [ ]:
# ==========================================
# 2. Data Generation
# ==========================================
# All lengths in micrometers. Same instrument as ch05-17.
lam1 = 0.55        # wavelength, green light
f_lens1 = 1.0e5    # focal length, 100 mm
diam1 = 2.5e4      # lens diameter, 25 mm

scale1 = lam1 * f_lens1 / diam1        # lambda f / D
r_airy1 = 1.2196699 * scale1           # Airy radius, the resolution yardstick


def jinc(u):
    """2 J1(u) / u, which equals 1 at u = 0."""
    u_safe = np.where(np.abs(u) < 1e-12, 1e-12, u)
    return 2 * j1(u_safe) / u_safe


def psf_intensity(rho):
    """Incoherent PSF |h|^2, the magnitude squared, normalized to 1."""
    return jinc(np.pi * diam1 * rho / (lam1 * f_lens1)) ** 2


# --- The object: point sources, already scaled by the magnification ---
src_pos1 = np.array([-20.0, -14.0, -9.0, -2.0, 4.0, 5.8, 17.0])
src_amp1 = np.array([0.40, 0.65, 1.00, 0.50, 0.75, 0.55, 0.30])

# --- The image plane ---
Y1 = np.linspace(-30.0, 30.0, 12001)
dY1 = Y1[1] - Y1[0]

# Each source paints its own copy of the PSF, scaled and shifted
parts1 = np.array([a * psf_intensity(Y1 - p)
                   for p, a in zip(src_pos1, src_amp1)])
image1 = parts1.sum(axis=0)            # the image is their sum

# The PSF on its own, for the middle panel
psf1 = psf_intensity(Y1)

# The close pair, for the annotation
close_gap1 = src_pos1[5] - src_pos1[4]


## Plot the image as a sum of Airy disks

Three panels, read downward, in the order the physics happens.

**The top panel is the object.** Each arrow is a point source, drawn as a delta
function because that is what a point source is: all its light at one position. The
arrow's height is the source's brightness.

**The middle panel is the PSF**, the single Airy disk that any one source produces.
This is the shape about to be stamped down at every arrow.

**The bottom panel is the image**, and it is drawn two ways at once. The faint dashed
curves are the individual contributions, one per source, each the PSF scaled by that
source's weight and shifted to its position. The solid curve is their sum, which is
what a detector actually records. Where the sources are far apart the two agree and
each source keeps its own peak. Where they crowd together the dashed curves overlap and
the solid one stops resolving them.

That is the whole convolution, drawn: **shift a copy of $\left\lvert h \right\rvert^2$ to every
source, scale it, and add.** The integral in the introduction is the continuous limit of this picture.

The marked pair near $5~\mu$m is the case to look at. Two separate sources, one bump.


In [ ]:
# ==========================================
# 3. Plotting the convolution
# ==========================================
fig1, axes1 = plt.subplots(3, 1, figsize=(13, 11))
plt.subplots_adjust(hspace=0.42, left=0.10, right=0.97, top=0.95, bottom=0.08)
ax_obj1, ax_psf1, ax_img1 = axes1

xlim1 = 28.0


def gutter_y(ax_top, ax_bot):
    """Vertical midpoint of the gap between two stacked axes, in figure coords."""
    return (ax_top.get_position().y0 + ax_bot.get_position().y1) / 2


# --- Top: the object, a row of weighted deltas ---
for p1, a1 in zip(src_pos1, src_amp1):
    ax_obj1.annotate("", xy=(p1, a1), xytext=(p1, 0),
                     arrowprops=dict(arrowstyle="-|>", lw=2.5, color='red',
                                     mutation_scale=18,
                                     shrinkA=0, shrinkB=0))
ax_obj1.axhline(0, color='black', linewidth=1, alpha=0.5)
ax_obj1.set_title(r"Object:  $I_\mathrm{obj}$, a sum of weighted point sources",
                  fontsize=16, color='red')
ax_obj1.set_ylabel("brightness")
ax_obj1.set_xlim(-xlim1, xlim1)
ax_obj1.set_ylim(-0.08, 1.20)
ax_obj1.tick_params(labelbottom=False)
apply_custom_formatting(ax_obj1)

# --- Middle: the PSF that each source paints ---
ax_psf1.plot(Y1, psf1, 'b')
ax_psf1.fill_between(Y1, 0, psf1, color='blue', alpha=0.12)
ax_psf1.set_title(r"System:  the incoherent PSF $\left| h \right|^2$", fontsize=16,
                  color='blue')
ax_psf1.set_ylabel(r"$\left| h \right|^2$")
ax_psf1.set_xlim(-xlim1, xlim1)
ax_psf1.set_ylim(-0.08, 1.20)
ax_psf1.tick_params(labelbottom=False)
ax_psf1.annotate(r"width $\sim r_A = %.2f\ \mu$m" % r_airy1,
                 xy=(r_airy1, 0.05), xytext=(11.0, 0.62),
                 arrowprops=dict(arrowstyle="->", color='blue', linewidth=1.6),
                 fontsize=14, color='blue', ha='center')
apply_custom_formatting(ax_psf1)

# --- Bottom: the image, contributions and their sum ---
for row1 in parts1:
    ax_img1.plot(Y1, row1, color='green', linestyle='--', linewidth=1.2, alpha=0.55)
ax_img1.plot(Y1, image1, 'g-', linewidth=2.8)
ax_img1.fill_between(Y1, 0, image1, color='green', alpha=0.10)
ax_img1.set_title(r"Image:  $I_\mathrm{obj} \otimes \left| h \right|^2$,  every copy added together",
                  fontsize=16, color='green')
ax_img1.set_xlabel(r"image plane coordinate $Y$  ($\mu$m)")
ax_img1.set_ylabel(r"$I_\mathrm{image}$")
ax_img1.set_xlim(-xlim1, xlim1)
ax_img1.set_ylim(-0.08, 1.20)
apply_custom_formatting(ax_img1)

# The pair that fails to resolve
mid1 = (src_pos1[4] + src_pos1[5]) / 2
bump_y1 = image1[np.argmin(np.abs(Y1 - mid1))]
ax_img1.annotate("two sources, one bump\n(%.1f $\\mu$m apart = %.2f $r_A$)"
                 % (close_gap1, close_gap1 / r_airy1),
                 xy=(mid1, bump_y1 + 0.03), xytext=(19.5, 0.95),
                 arrowprops=dict(arrowstyle="->", color='darkgreen', linewidth=1.8),
                 fontsize=13, color='darkgreen', ha='center')

# --- Operators in the gutters between the stacked panels ---
gy1 = gutter_y(ax_obj1, ax_psf1)
gy2 = gutter_y(ax_psf1, ax_img1)
# One x for word and symbol, so they cannot drift apart
op_x1 = 0.53
fig1.text(op_x1, gy1 + 0.030, "convolve", fontsize=15, ha='center', va='center')
fig1.text(op_x1, gy1 - 0.004, r"$\otimes$", fontsize=30, ha='center', va='center')
fig1.text(op_x1, gy2 + 0.030, "gives", fontsize=15, ha='center', va='center')
fig1.text(op_x1, gy2 - 0.004, r"$=$", fontsize=30, ha='center', va='center')

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.


In [ ]:
# fig1.savefig("../figures/ch05/ch05-18-deltas-to-airy.jpg", dpi=200, bbox_inches='tight')


## The Rayleigh criterion

How close is too close? The classical answer, due to Rayleigh, is to put the peak of one
Airy disk on the **first dark ring** of the other. That is a separation of exactly one
Airy radius,

$$\Delta_\textrm{min} = 1.22\,\frac{\lambda f}{D}$$

and it is a convention rather than a law of nature. What makes it a *useful* convention
is that it corresponds to a definite, repeatable feature: at that separation the sum of
the two disks dips to about **73.5%** of its peaks, and a dip of roughly a quarter is
about what an eye or a detector can be relied on to notice.

The figure shows three separations, the same two sources each time, with only the
spacing changed. The optics are identical in all three panels, so the PSF is the same
shape throughout and the spacing is the only variable.

Read the top row as what the detector sees and the bottom row as why. At
$2.0\,r_A$ the two disks barely touch. At $1.0\,r_A$ they merge into a single elongated
blob with a visible waist. At $0.6\,r_A$ the dip is gone entirely and no amount of
looking will recover it, because the information is not in the image any more.

Note what does **not** help: the sources are still two sources, and the image is still
an exact calculation. The limit is set by $\lambda f / D$, which is to say by the
aperture. Opening the aperture is the only thing that moves it.


In [ ]:
# ==========================================
# 4. Plotting the Rayleigh criterion
# ==========================================
# Three separations, measured in Airy radii. Only the spacing changes.
seps2 = np.array([2.0, 1.0, 0.6]) * r_airy1
labels2 = ["well resolved", "Rayleigh criterion", "not resolved"]

# 1D cut through both sources
Y2 = np.linspace(-9.0, 9.0, 4001)

# 2D field, for the images along the top row
half_x2, half_y2 = 9.0, 4.5
X2g = np.linspace(-half_x2, half_x2, 601)
Y2g = np.linspace(-half_y2, half_y2, 301)
XX2, YY2 = np.meshgrid(X2g, Y2g)
gamma2 = 0.4

fig2, axes2 = plt.subplots(2, 3, figsize=(16, 8),
                           gridspec_kw={'height_ratios': [1, 1.35]})
plt.subplots_adjust(wspace=0.22, hspace=0.30, left=0.07, right=0.97,
                    top=0.91, bottom=0.09)

for col2, (sep2, lab2) in enumerate(zip(seps2, labels2)):
    a2, b2 = -sep2 / 2, sep2 / 2

    # --- Top row: what the detector sees ---
    img2 = (psf_intensity(np.sqrt((XX2 - a2) ** 2 + YY2 ** 2))
            + psf_intensity(np.sqrt((XX2 - b2) ** 2 + YY2 ** 2)))
    ax_top2 = axes2[0, col2]
    ax_top2.imshow(img2 ** gamma2, extent=[-half_x2, half_x2, -half_y2, half_y2],
                   cmap='gray', origin='lower', vmin=0, vmax=2 ** gamma2)
    ax_top2.set_title("%s\n$\\Delta = %.2f\\ r_A = %.2f\\ \\mu$m"
                      % (lab2, sep2 / r_airy1, sep2), fontsize=15)
    ax_top2.set_yticks([])
    ax_top2.tick_params(which='major', direction='in', length=5, width=1,
                        color='white', top=True, labelsize=12)

    # --- Bottom row: why ---
    p1_2 = psf_intensity(Y2 - a2)
    p2_2 = psf_intensity(Y2 - b2)
    tot2 = p1_2 + p2_2

    ax_bot2 = axes2[1, col2]
    ax_bot2.plot(Y2, p1_2, color='gray', linestyle='--', linewidth=1.6)
    ax_bot2.plot(Y2, p2_2, color='gray', linestyle='--', linewidth=1.6)
    ax_bot2.plot(Y2, tot2, 'g-', linewidth=2.8)
    ax_bot2.fill_between(Y2, 0, tot2, color='green', alpha=0.10)

    # "Resolved" means the sum has a genuine local minimum between the sources.
    # The test is the curvature at the midpoint: positive means a valley,
    # negative means the two disks have merged into one peak.
    i0_2 = np.argmin(np.abs(Y2))
    center2 = tot2[i0_2]
    curv2 = tot2[i0_2 - 1] - 2 * center2 + tot2[i0_2 + 1]
    peak2 = tot2.max()

    if curv2 > 0:
        ax_bot2.annotate("", xy=(0, center2), xytext=(0, peak2),
                         arrowprops=dict(arrowstyle="<->", color='darkred',
                                         linewidth=1.6))
        ax_bot2.text(0, peak2 + 0.13, "dip to %.1f%%" % (100 * center2 / peak2),
                     fontsize=14, color='darkred', ha='center', va='bottom')
    else:
        ax_bot2.text(0, peak2 + 0.13, "no dip: one peak", fontsize=14,
                     color='darkred', ha='center', va='bottom')

    for pos2 in (a2, b2):
        ax_bot2.plot(pos2, 0, 'v', color='red', markersize=9, zorder=6)

    ax_bot2.set_xlim(-half_x2, half_x2)
    ax_bot2.set_ylim(-0.10, 1.78)
    ax_bot2.set_xlabel(r"$Y$  ($\mu$m)")
    if col2 == 0:
        ax_bot2.set_ylabel(r"$I_\mathrm{image}$")
    apply_custom_formatting(ax_bot2)

plt.show()


### Save plot

Commented out on purpose. Uncomment it to regenerate the figure used in the chapter;
leaving it live would rewrite that file every time the notebook is run.


In [ ]:
# fig2.savefig("../figures/ch05/ch05-18-rayleigh.jpg", dpi=200, bbox_inches='tight')
